In [ ]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI
import os
from urllib.request import urlopen, Request
import base64
from pathlib import Path
from dotenv import load_dotenv

# Add references
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

In [ ]:
# Get configuration settings 
load_dotenv()
openai_endpoint = os.getenv("ENDPOINT")
model_deployment =  os.getenv("MODEL_DEPLOYMENT")


# Create an OpenAI client
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://ai.azure.com/.default")
client = OpenAI(
    base_url=openai_endpoint,
    api_key=token_provider()
) 
chat_completion = client.chat.completions.create(
    model=model_deployment,
    messages=[{"role": "user", "content": "who is prime minester of india"}]
) 
print(chat_completion.choices[0].message.content)  

In [ ]:
import json

# Example dummy function hard coded to return the same weather
# In production, this could be your backend API or an external API
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    weather_info = {
        "location": location,
        "temperature": "72",
        "unit": unit,
        "forecast": ["sunny", "windy"],
    }
    return json.dumps(weather_info)

In [ ]:
# define a function
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    }
]

In [ ]:
messages = [
    {
        "role": "user",
        "content": "What's the weather like in San Francisco?"
    }
]

In [ ]:
# Call the ChatCompletion endpoint
response = client.chat.completions.create(
    # OpenAI Updates: As of June 2024, we are now using the GPT-3.5-Turbo model
    model=model_deployment,
    messages=messages,
    functions=functions,
     function_call={"name": "get_current_weather"}
)

In [ ]:
print(response)

In [ ]:
response_message = response.choices[0].message

In [ ]:
content = response_message.content
print(content)

In [ ]:
function_cal=response_message.function_call
print(function_cal)

In [ ]:
json.loads(response_message.function_call.arguments)

In [ ]:
args=json.loads(response_message.function_call.arguments)

In [ ]:
get_current_weather(args)

LangChain starts

In [ ]:
%pip install langchain

In [ ]:
%pip install -U langchain langchain-core langchain-openai

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "tell me a short joke about {topic}"
)
model = ChatOpenAI(base_url=openai_endpoint,
                   model=model_deployment,
                   api_key=token_provider())
output_parser = StrOutputParser()

In [ ]:
chain = prompt | model | output_parser

In [ ]:
chain.invoke({"topic": "bears"})

In [ ]:
%pip install -U \
langchain \
langchain-core \
langchain-openai \
langchain-community \
docarray

In [ ]:
import langchain
import langchain_openai
import langchain_community

print(langchain.__version__)

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import DocArrayInMemorySearch

In [ ]:
embedding_deployment = os.getenv("EMBEDDING_MODEL_DEPLOYMENT")

In [ ]:
vectorstore = DocArrayInMemorySearch.from_texts(
    ["harrison worked at kensho", "bears like to eat honey"],
    embedding=OpenAIEmbeddings(base_url=openai_endpoint,
                   model=embedding_deployment,
                   api_key=token_provider())
)
retriever = vectorstore.as_retriever()

In [ ]:
retriever.invoke("where did harrison work?")

In [ ]:
retriever.invoke("what do bears like to eat")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
from langchain_core.runnables import RunnableMap

In [62]:
from langchain_core.runnables import RunnableMap

chain = (
    RunnableMap({
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"]
    })
    | prompt
    | model
    | StrOutputParser()
)

In [63]:
chain.invoke({
    "question": "where did harrison work?"
})

'Harrison worked at Kensho.'

In [61]:
chain = (
    RunnableMap({
        "context": retriever,
        "question": lambda x: x
    })
    | prompt
    | model
    | StrOutputParser()
)

chain.invoke("where did harrison work?")

'Harrison worked at Kensho.'

In [66]:
functions = [
    {
      "name": "weather_search",
      "description": "Search for weather given an airport code",
      "parameters": {
        "type": "object",
        "properties": {
          "airport_code": {
            "type": "string",
            "description": "The airport code to get the weather for"
          },
        },
        "required": ["airport_code"]
      }
    }
  ]

In [68]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}")
    ]
)
model = ChatOpenAI(base_url=openai_endpoint,
                   model=model_deployment,
                   api_key=token_provider()).bind(functions=functions)

In [69]:
runnable = prompt | model

In [70]:
runnable.invoke({"input": "what is the weather in sf"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"SFO"}', 'name': 'weather_search'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 143, 'total_tokens': 168, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 8, 'engine_ttft_ms': 25, 'engine_ttlt_ms': 521, 'pre_inference_ms': 204, 'service_tbt_ms': 8, 'service_ttft_ms': 581, 'service_ttlt_ms': 1072, 'user_visible_ttft_ms': 377}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ES2Iibrdwydt23N8mezwaMztSIOPu', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='lc_run--01a0d938-3515-7681-90a8-79bd4d245f1b-0', tool_calls=[], invalid_tool_cal

In [71]:
functions = [
    {
      "name": "weather_search",
      "description": "Search for weather given an airport code",
      "parameters": {
        "type": "object",
        "properties": {
          "airport_code": {
            "type": "string",
            "description": "The airport code to get the weather for"
          },
        },
        "required": ["airport_code"]
      }
    },
        {
      "name": "sports_search",
      "description": "Search for news of recent sport events",
      "parameters": {
        "type": "object",
        "properties": {
          "team_name": {
            "type": "string",
            "description": "The sports team to search for"
          },
        },
        "required": ["team_name"]
      }
    }
  ]

In [72]:
model = model.bind(functions=functions)

In [73]:
runnable = prompt | model

In [74]:
runnable.invoke({"input": "how did the patriots do yesterday?"})

AIMessage(content='Do you mean the New England Patriots (NFL) or a different "Patriots" team? Also do you want the final score only or a short recap of the game and key stats?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 176, 'prompt_tokens': 176, 'total_tokens': 352, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 8, 'engine_ttft_ms': 24, 'engine_ttlt_ms': 1708, 'pre_inference_ms': 93, 'service_tbt_ms': 8, 'service_ttft_ms': 450, 'service_ttlt_ms': 2086, 'user_visible_ttft_ms': 357}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ES2JtTB6TnBD5YMhA37MRiGIva8Lo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0d9